# Đồ Án Cuối Kỳ - Topic 6: Real-Time Weather Data Collection and Cleaning
## Tích hợp Đa nguồn từ 4 Web API Thực tế (Open-Meteo SDK + WeatherAPI.com + Xweather API + FreeOpenAPI.dev)

**Mục tiêu đồ án:** Thu thập dữ liệu quan trắc thời tiết thực tế 100% từ **4 nguồn Web API độc lập** trên **20 tọa độ địa điểm trải dài khắp Việt Nam**, xây dựng quy trình làm sạch, chuẩn hóa và hợp nhất đa cảm biến (**Multi-API Sensor Fusion & Time-Series Cleaning Pipeline**), đánh giá chất lượng dữ liệu trước/sau khi làm sạch và phân tích trực quan.

### 4 Nguồn Web API Thực tế được Tích hợp:
1. **Open-Meteo Official Python SDK (`openmeteo_requests`):** `44.640` bản ghi (`past_days=92, forecast_days=1`, 20 địa điểm $\times$ 2.232 giờ).
2. **WeatherAPI.com (`/v1/forecast.json` + `current`):** `1.460` bản ghi (20 địa điểm $\times$ 72 giờ dự báo/quan trắc + 20 bản ghi thời gian thực).
3. **Vaisala Xweather API (`/conditions`):** `480` bản ghi (20 địa điểm $\times$ 24 giờ quan trắc khí tượng gần nhất).
4. **FreeOpenAPI.dev (`/api/v1/weather/current`):** `20` bản ghi thời gian thực tại 20 địa điểm.
- **Tổng quy mô thu thập thô (`df_raw`):** **`46.600` quan trắc thực tế** $\rightarrow$ Làm sạch và hợp nhất thành **`30.220` quan trắc chuỗi thời gian liên tục** (`20 địa điểm × 1.511 giờ`, trong đó có **`740` mốc giờ được tổng hợp chéo từ $\ge 2$ API**).

In [ ]:
import json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

BASE_DIR = Path.cwd()
RAW_DIR = BASE_DIR / "data" / "raw"
CLEAN_DIR = BASE_DIR / "data" / "cleaned"
FIG_DIR = BASE_DIR / "figures"

# Đọc dữ liệu thô thu thập từ 4 nguồn Web API thực tế
df_raw = pd.read_csv(RAW_DIR / "raw_weather_data.csv")
print(f"Kích thước tập dữ liệu thô (df_raw): {df_raw.shape[0]:,} dòng x {df_raw.shape[1]} cột")
print("\nSố lượng bản ghi theo từng nguồn Web API:")
print(df_raw["source_api"].value_counts())
df_raw.groupby("source_api").head(2)

### Phân tích kết quả Thu thập Đa nguồn từ 4 Web API Thực tế

Khi kết hợp dữ liệu thực từ 4 nhà cung cấp API khác nhau (`Open-Meteo SDK`, `WeatherAPI.com`, `Xweather API`, `FreeOpenAPI.dev`) trên 20 địa điểm tại Việt Nam (`46.600` dòng thô), toàn bộ các vấn đề kinh điển của bài toán **Data Cleaning & Integration** đều xuất hiện hoàn toàn tự nhiên:
1. **4 Định dạng thời gian (`date`) khác nhau giữa 4 API (`46.600` dòng):**
   - `Open-Meteo SDK`: `"2026-10-17 11:00:00+07:00"` (có khoảng trắng, có giây, có hậu tố `+07:00`).
   - `WeatherAPI.com`: `"2026-10-17 11:00"` và `"2026-10-17 11:30"` (không có giây, không có hậu tố múi giờ).
   - `Xweather API`: `"2026-10-16T11:00:00+07:00"` (chuẩn ISO ngăn cách bằng chữ `T` kèm `+07:00`).
   - `FreeOpenAPI.dev`: `"2026-10-17T11:30:00"` (chuẩn ISO có chữ `T`, không có hậu tố múi giờ, lấy mẫu ở phút `:30`).
2. **Sự thiếu hụt thuộc tính do khác biệt Schema & Giới hạn lưu trữ (`92.340` ô `NaN`):**
   - `Open-Meteo SDK` trả về `15.380` dòng rỗng hoàn toàn (`92.280` ô `NaN`) ở 32 ngày đầu tiên (`2026-07-17` $\rightarrow$ `2026-08-18`) do giới hạn bộ đệm 61 ngày của endpoint `/v1/forecast`.
   - `FreeOpenAPI.dev` chỉ trả về nhiệt độ, tốc độ gió và lượng mưa trong khối `/current`, dẫn đến **khuyết `60` ô (`20` dòng $\times$ 3 cột `relative_humidity_2m`, `pressure_msl`, `wind_gusts_10m`)**.
3. **Trùng lặp quan trắc đa nguồn (`1.000` bản ghi trùng khóa `(location_id, time)`):**
   - Trong khung giờ giao thoa giữa 4 API (72 giờ của WeatherAPI, 24 giờ của Xweather, giờ hiện tại của FreeOpenAPI và Open-Meteo), mỗi địa điểm tại cùng một mốc giờ có từ **2 đến 4 bản ghi** từ các API khác nhau.

In [ ]:
# Kiểm tra chi tiết Missing API Responses, bất nhất vật lý và ngoại lai thống kê (IQR) trên dữ liệu thật
WEATHER_VARS = [
    "temperature_2m", "wind_speed_10m", "relative_humidity_2m",
    "precipitation", "pressure_msl", "wind_gusts_10m"
]

print("=== 1. SỐ LƯỢNG GIÁ TRỊ KHUYẾT (NaN) TRONG DỮ LIỆU THÔ TỪ API ===")
print(df_raw[WEATHER_VARS].isna().sum())

valid_mask = ~df_raw[WEATHER_VARS].isna().all(axis=1)
df_active = df_raw[valid_mask].copy()
print(f"\n- Số dòng API rỗng hoàn toàn (2026-07-17 -> 2026-08-18 00:00) : {(~valid_mask).sum():,} dòng")
print(f"- Số dòng API có dữ liệu hợp lệ (2026-08-18 01:00 -> 2026-10-17): {len(df_active):,} dòng")

wind_inconsistent = df_active[df_active["wind_gusts_10m"] < df_active["wind_speed_10m"]]
print(f"- Số dòng bất nhất vật lý (wind_gusts_10m < wind_speed_10m)   : {len(wind_inconsistent):,} dòng")

### Phân tích kết quả Kiểm tra Phản hồi Khuyết (Missing API Responses) & Tính Nhất quán Vật lý

Phân tích trực tiếp trên 44.640 dòng dữ liệu thực từ Open-Meteo SDK cho thấy:
1. **Phát hiện vùng khuyết dữ liệu của API (`15.380` dòng / `92.280` ô `NaN`):**
   - Toàn bộ 6 biến khí tượng đều có đúng **15.380 giá trị `NaN`** nằm tập trung từ `2026-07-17 00:00:00+07:00` đến `2026-08-18 00:00:00+07:00` (769 giờ đầu tiên của mỗi địa điểm).
   - **Quyết định xử lý:** Nếu áp dụng `bfill()` cho 769 giờ đầu tiên này, dữ liệu sẽ bị biến dạng thành một đường ngang cố định suốt 32 ngày. Do đó, phương pháp đúng đắn về mặt Khoa học Dữ liệu là **cắt bỏ vùng chân trời rỗng của API (`15.380` dòng)** và giữ lại cửa sổ chuỗi thời gian liên tục **29.260 dòng** (`1.463` giờ/địa điểm từ `2026-08-18 01:00` đến `2026-10-17 23:00`), nơi không có bất kỳ ô `NaN` lẻ tẻ nào (`0` partial NaNs).
2. **Phát hiện bất nhất vật lý giữa Gió giật và Tốc độ gió (`94` dòng):**
   - Về mặt vật lý khí tượng, tốc độ gió giật cực đại trong giờ (`wind_gusts_10m`) luôn phải lớn hơn hoặc bằng tốc độ gió trung bình 10m (`wind_speed_10m`). Tuy nhiên, do bước nội suy mô hình số trị khác nhau giữa 2 biến trên Open-Meteo, có **94 bản ghi thực tế** có `wind_gusts_10m < wind_speed_10m`. Pipeline làm sạch đã hiệu chỉnh bằng quy tắc `wind_gusts_10m = max(wind_gusts_10m, wind_speed_10m)`.

In [ ]:
# Tải dữ liệu sau khi làm sạch (Cleaned Dataset), bảng đối sánh 4 API và Báo cáo chất lượng
df_clean = pd.read_csv(CLEAN_DIR / "cleaned_weather_data.csv")
df_overlap = pd.read_csv(CLEAN_DIR / "multi_source_comparison.csv")
with open(CLEAN_DIR / "data_quality_report.json", "r", encoding="utf-8") as f:
    quality_report = json.load(f)

comparison_df = pd.DataFrame({
    "Chỉ số chất lượng (Quality Metric)": [
        "Tổng số dòng dữ liệu (Total Rows)",
        "Số lượng địa điểm (Locations)",
        "Bản ghi chưa đồng nhất định dạng thời gian (4 API Formats)",
        "Số dòng API rỗng hoàn toàn (Empty API Horizon Rows >61d)",
        "Số dòng khuyết cục bộ do khác Schema (FreeOpenAPI Partial NaN Rows)",
        "Tổng số ô giá trị bị khuyết (Total Missing Numeric Cells - NaN)",
        "Bản ghi trùng khóa (location_id, time) giữa 4 API (Overlapping Duplicates)",
        "Số ô bị nhiễu dấu chấm động Float32 (>2 chữ số thập phân)",
        "Bản ghi bất nhất vật lý (wind_gusts_10m < wind_speed_10m)",
    ],
    "Trước khi làm sạch (Raw 4-API Data)": [
        quality_report["before_cleaning"]["total_rows"],
        quality_report["before_cleaning"]["num_locations"],
        quality_report["before_cleaning"]["unstandardized_timestamps"],
        quality_report["before_cleaning"]["empty_api_response_rows"],
        quality_report["before_cleaning"]["partial_missing_rows"],
        quality_report["before_cleaning"]["total_missing_cells"],
        quality_report["before_cleaning"]["key_duplicates"],
        quality_report["before_cleaning"]["float32_precision_artifacts"],
        quality_report["before_cleaning"]["wind_gust_inconsistencies"],
    ],
    "Sau khi làm sạch (Cleaned Data)": [
        quality_report["after_cleaning"]["total_rows"],
        quality_report["after_cleaning"]["num_locations"],
        quality_report["after_cleaning"]["unstandardized_timestamps"],
        quality_report["after_cleaning"]["empty_api_response_rows"],
        quality_report["after_cleaning"]["partial_missing_rows"],
        quality_report["after_cleaning"]["total_missing_cells"],
        quality_report["after_cleaning"]["key_duplicates"],
        quality_report["after_cleaning"]["float32_precision_artifacts"],
        quality_report["after_cleaning"]["wind_gust_inconsistencies"],
    ]
})
display(comparison_df)
df_clean[df_clean["num_sources"] >= 3].head(8)

### Phân tích kết quả Đối sánh Chất lượng Dữ liệu Trước và Sau khi Làm sạch

Bảng đối sánh chứng minh hiệu quả làm sạch trên dữ liệu thực tế từ `openmeteo_requests`:
- **Loại bỏ triệt để phản hồi API bị rỗng (`15.380` dòng / `92.280` ô `NaN`):** Các dòng vượt quá giới hạn lưu trữ 61 ngày của endpoint `/v1/forecast` đã được phát hiện và lọc bỏ chính xác, giữ lại **29.260 dòng quan trắc liên tục** (`1.463` giờ/địa điểm từ `2026-08-18 01:00:00` đến `2026-10-17 23:00:00`) với **0 ô bị khuyết (`0%` missing)**.
- **Chuẩn hóa định dạng thời gian và dấu chấm động `Float32`:** Toàn bộ `44.640` mốc thời gian có hậu tố `+07:00` và `72.418` ô số thực bị nhiễu thập phân nhị phân (`float32` sang `float64`) đã được chuẩn hóa về định dạng ISO `YYYY-MM-DD HH:MM:SS` và làm tròn chuẩn xác đến 2 chữ số thập phân.
- **Đảm bảo tính nhất quán vật lý 100%:** Toàn bộ **94 dòng** có `wind_gusts_10m < wind_speed_10m` đã được hiệu chỉnh, và 100% giá trị nằm trong miền đo lường khí tượng hợp lệ.

In [ ]:
# Thống kê mô tả (Descriptive Statistics) cho toàn bộ 20 địa điểm sau khi làm sạch
city_summary = pd.read_csv(CLEAN_DIR / "city_summary_statistics.csv")
display(city_summary)

# Hiển thị biểu đồ đối sánh chất lượng dữ liệu và Boxplot kiểm tra miền giá trị
from IPython.display import Image, display as ipy_display
ipy_display(Image(filename=str(FIG_DIR / "fig1_data_quality_before_after.png")))
ipy_display(Image(filename=str(FIG_DIR / "fig2_range_validation_boxplots.png")))

### Phân tích Thống kê Mô tả 20 Địa điểm (1.463 Giờ/Địa điểm) & Kiểm định Ngoại lai Khí tượng Thực tế

1. **Phát hiện các hiện tượng thời tiết cực đoan thực tế (`fig1` & `fig2`):**
   - Khác với lỗi cảm biến giả lập, khi kiểm tra phân phối của **29.260 quan trắc thực tế** bằng quy tắc tứ phân vị ($1.5 \times \text{IQR}$ theo từng địa điểm), chúng ta phát hiện **1.495 giờ có thời tiết cực đoan thực sự** trong 2 tháng qua:
     - `wind_speed_10m`: **434 giờ** gió mạnh vượt ngưỡng IQR (đặc biệt tại Côn Đảo đạt cực đại `41.60 km/h`, Vinh `38.01 km/h`, Pleiku `35.69 km/h`).
     - `precipitation`: **355 giờ** mưa lớn (`>5.0 mm/h`), với đỉnh mưa theo giờ tại Hạ Long (`37.40 mm/h`) và Phú Lộc - Huế (`37.20 mm/h`).
     - `wind_gusts_10m`: **340 giờ** gió giật mạnh, đạt đỉnh tại Pleiku (`81.00 km/h`), Vũng Tàu (`75.24 km/h`), và Vinh (`71.64 km/h`).
     - `temperature_2m`: **171 giờ** nhiệt độ cực trị, từ mức lạnh nhất tại Sa Pa (`11.40°C`) đến mức nóng nhất tại Đà Nẵng (`39.70°C`) và TP. Huế (`38.90°C`).
2. **Sự phân hóa theo độ cao và vùng miền (`city_summary_statistics.csv`):**
   - `Sa Pa` (độ cao `1545m`) có nhiệt độ trung bình thấp nhất cả nước (**`18.60°C`**), tiếp theo là `Phu Loc - Hue` (`1002m`, **`21.34°C`**), `Pleiku` (`761m`, **`22.69°C`**) và `Da Lat - Lam Dong` (`707m`, **`23.00°C`**).
   - Các trạm duyên hải và đồng bằng có nền nhiệt trung bình cao nhất: `Nha Trang` (**`28.54°C`**), `Da Nang` (**`28.43°C`**), `Con Dao` (**`28.18°C`**) và `Vung Tau` (**`28.18°C`**).

In [ ]:
# Hiển thị các biểu đồ phân tích chuỗi thời gian, so sánh 20 thành phố, tương quan đa biến và đối sánh 4 API
ipy_display(Image(filename=str(FIG_DIR / "fig3_temperature_timeseries.png")))
ipy_display(Image(filename=str(FIG_DIR / "fig4_all_20_cities_temperature_range.png")))
ipy_display(Image(filename=str(FIG_DIR / "fig5_multivariate_and_precipitation.png")))
ipy_display(Image(filename=str(FIG_DIR / "fig6_multi_api_source_comparison.png")))

### Phân tích Xu hướng Chuỗi Thời gian 2 Tháng, Biên độ Nhiệt & Tương quan Khí tượng

1. **Xu hướng biến thiên nhiệt độ qua 61 ngày liên tục (`fig3_temperature_timeseries.png`):**
   - Biểu đồ chuỗi thời gian trung bình ngày từ `2026-08-18` đến `2026-10-17` cho thấy xu hướng chuyển mùa rõ rệt từ cuối mùa hè (tháng 8) sang mùa thu (tháng 10), đặc biệt tại các tỉnh miền Bắc và vùng núi (`Hanoi`, `Sa Pa`) và miền Trung (`Da Nang`) khi nền nhiệt giảm dần và xuất hiện các đợt không khí lạnh/mưa lớn làm nhiệt độ giảm sâu theo từng đợt.
2. **Biên độ nhiệt giữa 20 tọa độ khảo sát (`fig4_all_20_cities_temperature_range.png`):**
   - Hải đảo xa bờ như `Con Dao` có độ lệch chuẩn nhiệt độ thấp nhất cả nước ($\sigma = 0.90^\circ\text{C}$, dao động chỉ từ `25.5°C – 30.8°C` trong suốt 2 tháng).
   - Ngược lại, các tỉnh miền Trung chịu hiệu ứng gió Phơn và mưa thu như `Da Nang` (`23.65°C – 39.70°C`), `Hue City` (`22.45°C – 38.90°C`) và vùng núi Tây Bắc như `Dien Bien Phu` (`17.74°C – 33.94°C`) có biên độ nhiệt rất rộng.
3. **Tương quan đa biến và tổng lượng mưa tích lũy (`fig5_multivariate_and_precipitation.png`):**
   - Biểu đồ phân tán khẳng định quy luật vật lý chặt chẽ giữa **Nhiệt độ – Độ ẩm – Áp suất khí quyển**: nhiệt độ tỷ lệ nghịch với độ ẩm tương đối, trong khi các điểm có áp suất mực nước biển cao (`>1018 hPa`) tập trung ở vùng nhiệt độ thấp (`<20°C` tại Sa Pa / vùng núi phía Bắc).
   - Về tổng lượng mưa tích lũy trong 61 ngày (`1.463` giờ), khu vực **Bắc Trung Bộ, Trung Trung Bộ và Tây Nguyên** dẫn đầu cả nước: `Vinh` (**`934.4 mm`**), `Phu Loc - Hue` (**`927.1 mm`**), `Nam Dinh` (**`864.9 mm`**), `Pleiku` (**`801.1 mm`**), và `Da Nang` (**`798.6 mm`**).

---

## Tổng Kết (Conclusion & Comprehensive Summary)

Đồ án đã hoàn thành toàn diện mọi yêu cầu của **Topic 6: Real-Time Weather Data Collection and Cleaning** trên bộ dữ liệu thực 100% từ **Open-Meteo Python SDK (`openmeteo_requests`)**:
- **Thu thập dữ liệu thực tế (`src/01_collect_weather_data.py`):** Sử dụng `openmeteo_requests.Client` với `requests_cache` và `retry_requests` lấy dữ liệu của **20 tọa độ tại Việt Nam** (`past_days=92`, `forecast_days=1`), thu về **44.640 dòng dữ liệu thô** với 6 biến khí tượng (`temperature_2m`, `wind_speed_10m`, `relative_humidity_2m`, `precipitation`, `pressure_msl`, `wind_gusts_10m`).
- **Làm sạch & Chuẩn hóa Chuỗi thời gian (`src/02_clean_and_preprocess.py`):**
  1. Phát hiện và cắt bỏ **15.380 dòng phản hồi rỗng (`92.280` ô `NaN`)** do giới hạn lưu trữ ~61 ngày của endpoint `/v1/forecast`.
  2. Chuẩn hóa 100% định dạng thời gian về ISO `YYYY-MM-DD HH:MM:SS` (`Asia/Bangkok`, UTC+7) và khử nhiễu biểu diễn nhị phân `float32` trên **72.418 ô dữ liệu**.
  3. Kiểm tra trùng lặp, kiểm định tính liên tục của lưới thời gian (`1.463` giờ/địa điểm), sửa **94 bản ghi bất nhất vật lý** (`wind_gusts_10m < wind_speed_10m`) và phân tích **1.495 giờ thời tiết cực đoan thực tế** theo IQR, xuất bộ dữ liệu sạch hoàn chỉnh **29.260 dòng** tại `data/cleaned/cleaned_weather_data.csv`.